In [ ]:
%pip install earthscope-sdk earthscope-cli obspy pandas tqdm --quiet

In [ ]:
!es login

In [ ]:
import boto3
import pandas as pd
from botocore.config import Config
from earthscope_sdk import EarthScopeClient

BUCKET = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
PREFIX = "miniseed/"
WORKERS = 20


def refresh_s3_client(es_client, max_pool=WORKERS):
    creds = es_client.user.get_aws_credentials()
    def _v(x):
        return x.get_secret_value() if hasattr(x, 'get_secret_value') else x
    session = boto3.Session(
        aws_access_key_id=creds.aws_access_key_id,
        aws_secret_access_key=_v(creds.aws_secret_access_key),
        aws_session_token=_v(creds.aws_session_token),
    )
    s3_config = Config(
        request_checksum_calculation="when_required",
        response_checksum_validation="when_required",
        max_pool_connections=max_pool,
    )
    return session.client("s3", config=s3_config)


print("Authenticating with EarthScope...")
es_client = EarthScopeClient()
s3_client = refresh_s3_client(es_client)
print("Authenticated successfully.")

In [ ]:
from obspy.clients.fdsn import Client as FDSNClient

print("Fetching network list from S3...")
resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
network_list = [p["Prefix"].split("/")[1] for p in resp.get("CommonPrefixes", [])]
print(f" -> Found {len(network_list)} networks in S3.")

print("\nQuerying FDSN for network descriptions to identify synthetic/test networks...")
client_fdsn = FDSNClient("https://service.earthscope.org")

SYNTHETIC_KEYWORDS = {"synthetic", "simulated", "test", "artificial", "benchmark", "fictitious"}

# Real science networks whose descriptions happen to contain a keyword — keep these
WHITELIST = {
    "RS",  # RSTN — real historical DoD seismic monitoring network
    "XA",  # CTEMPs DAS/DTS — real fiber optic seismic deployments
    "YV",  # HFTS-EOR — real induced seismicity from hydraulic fracturing
    "YY",  # Mendenhall Glacier — real glacier seismicity + instrument science
    "ZN",  # Breidamerkurjokull — real subglacial hydrology seismicity
    "4E",  # MARS OBS — real cabled ocean bottom seismometer data
    "X1",  # Nootka — real deep-water acoustically-linked buoy observatory
    "ZT",  # ALST — real deep-water buoy observatory
}

to_drop = set()

try:
    inv = client_fdsn.get_stations(network=",".join(network_list), level="network")
    for net in inv:
        if net.code in WHITELIST:
            continue
        desc = (net.description or "").lower()
        if any(kw in desc for kw in SYNTHETIC_KEYWORDS):
            to_drop.add(net.code)
            print(f" -> Flagged '{net.code}': {net.description}")
except Exception as e:
    print(f"  -> Could not fetch network descriptions: {e}. Falling back to known list only.")

to_drop.add("SY")  # always drop — confirmed synthetic
to_drop -= WHITELIST  # safety: never drop whitelisted networks

for net in sorted(to_drop):
    if net in network_list:
        network_list.remove(net)

print(f"\nDropped {len(to_drop)} synthetic/test networks: {sorted(to_drop)}")
print(f"Kept (whitelisted): {sorted(WHITELIST & set(network_list))}")
print(f" -> {len(network_list)} networks remaining for S3 scan.")

In [ ]:
import time
from obspy.clients.fdsn import Client

print(f"Fetching FDSN metadata for {len(network_list)} networks...")
client_iris = Client("https://service.earthscope.org")
station_data = []
batch_size = 50

for i in range(0, len(network_list), batch_size):
    batch = network_list[i : i + batch_size]
    try:
        inventory = client_iris.get_stations(network=",".join(batch), level="station")
        for network in inventory:
            for station in network:
                end_time = station.end_date.datetime if station.end_date else pd.Timestamp("2100-01-01")
                station_data.append({
                    "network": network.code,
                    "station": station.code,
                    "lat": station.latitude,
                    "lon": station.longitude,
                    "start_date": station.start_date.datetime,
                    "end_date": end_time,
                })
    except Exception as e:
        print(f"  -> Batch {i // batch_size + 1} failed: {e}")
    time.sleep(1)

df_coords = pd.DataFrame(station_data)
df_coords["start_date"] = pd.to_datetime(df_coords["start_date"])
df_coords["end_date"] = pd.to_datetime(df_coords["end_date"])
print(f" -> Retrieved coordinates for {len(df_coords):,} stations.")

In [ ]:
import os
import gc
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

OUTPUT_FILE = "/content/s3_inventory.csv"
AUTH_LOCK = threading.Lock()
WORKERS = 20


def scan_network_records(network):
    global s3_client
    records = []
    paginator = s3_client.get_paginator("list_objects_v2")

    def _scan(pag):
        rows = []
        for page in pag.paginate(Bucket=BUCKET, Prefix=f"{PREFIX}{network}/"):
            for obj in page.get("Contents", []):
                parts = obj["Key"].split("/")
                if len(parts) >= 5:
                    rows.append((network, parts[-1].split(".")[0], parts[2], parts[3]))
        return rows

    try:
        records = _scan(paginator)
    except Exception as e:
        if "ExpiredToken" in str(e) or "Token expired" in str(e) or "Forbidden" in str(e):
            with AUTH_LOCK:
                s3_client = refresh_s3_client(es_client)
            try:
                records = _scan(s3_client.get_paginator("list_objects_v2"))
            except Exception as e2:
                tqdm.write(f"  -> ERROR {network}: {e2}")
        else:
            tqdm.write(f"  -> ERROR {network}: {e}")

    return records


all_records = []

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(scan_network_records, net): net for net in network_list}
    for future in tqdm(as_completed(futures), total=len(network_list), desc="Scanning networks"):
        all_records.extend(future.result())

print(f"\nScan complete — {len(all_records):,} raw file records.")
print("Merging with FDSN coords and aggregating...")

df_all = pd.DataFrame(all_records, columns=["network", "station", "year", "yearday"])
df_all["data_date"] = pd.to_datetime(
    df_all["year"] + df_all["yearday"].str.zfill(3), format="%Y%j"
)
df_merged = pd.merge(df_all, df_coords, on=["network", "station"], how="inner")
df_valid = df_merged[
    (df_merged["data_date"] >= df_merged["start_date"]) &
    (df_merged["data_date"] <= df_merged["end_date"])
]
df_stations = (
    df_valid.groupby(["network", "station", "lat", "lon"])
    .size()
    .reset_index(name="days")
)

df_stations.to_csv(OUTPUT_FILE, index=False)
print(f"Done! {len(df_stations):,} unique stations saved to {OUTPUT_FILE}.")

del df_all, df_merged, df_valid
gc.collect()

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)